# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: bật Internet trong Settings.
!pip -q install timm openpyxl

import sys, os, platform
import numpy as np, pandas as pd, torch, timm

# Đã `git clone` repo bài lab vào thư mục này (đổi nếu bạn đặt chỗ khác).
REPO_DIR = "K4-Track4-Day2-Deeplearning-Advance" if os.path.isdir("K4-Track4-Day2-Deeplearning-Advance") else "."
sys.path.insert(0, REPO_DIR)                 # để import eval.py
sys.path.insert(0, f"{REPO_DIR}/starter")    # hoặc thư mục code/ của bạn sau khi chép starter vào
if os.path.isdir("starter"):
    sys.path.insert(0, "starter")

print("python", platform.python_version(), "| torch", torch.__version__, "| timm", timm.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
**Sau khi giải nén, hãy `ls` để xem ảnh nằm ở thư mục nào** rồi đặt `IMAGES_DIR` cho đúng.

In [ ]:
import hashlib

os.makedirs("data/labels", exist_ok=True)
if not os.path.exists("data/images.zip"):
    !wget -q -O data/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1"

h = hashlib.md5()
with open("data/images.zip", "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"

!unzip -q -n data/images.zip -d data/
!ls data | head

BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    !wget -q -O data/labels/{name}.csv {BASE}/{name}.csv
!ls -la data/labels

In [ ]:
IMAGES_DIR = "data/images"   # TODO: kiểm tra đúng thư mục chứa các file .jpg sau khi giải nén
LABELS_DIR = "data/labels"

# Tự động phát hiện thư mục images/labels nếu chạy ở local repo
if not os.path.isdir(IMAGES_DIR) and os.path.isdir("images"):
    IMAGES_DIR = "images"
if not os.path.isdir(LABELS_DIR) and os.path.isdir("labels"):
    LABELS_DIR = "labels" 

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.

In [ ]:
# TODO: đọc split bằng dataset.load_split, chạy dataset.check_split(...), in số ảnh mỗi tập và mỗi lớp.
# TODO: vẽ biểu đồ phân bố lớp; xem >= 3 ảnh mỗi lớp; đối chiếu với Table 1 của bài báo
#       (Negatives 9.106; các loài 1.009-1.125 ảnh).
import matplotlib.pyplot as plt
from dataset import CLASS_NAMES, DeepWeedsDataset, build_transforms, check_split, load_split

# 1. Đọc và kiểm tra các split fold 0
train_df, val_df, test_df, meta_df = load_split(LABELS_DIR)
check_split(train_df, val_df, test_df, meta_df)

print(f"Tổng số ảnh: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

# 2. Phân bố nhãn giữa các tập
dist_df = pd.DataFrame({
    "Class": CLASS_NAMES,
    "Train": train_df["Label"].value_counts().sort_index(),
    "Val": val_df["Label"].value_counts().sort_index(),
    "Test": test_df["Label"].value_counts().sort_index(),
    "Total": meta_df["Label"].value_counts().sort_index() if meta_df is not None else 0
})
display(dist_df)

# 3. Trực quan hoá phân bố lớp
plt.figure(figsize=(10, 4))
dist_df.set_index("Class")[["Train", "Val", "Test"]].plot(kind="bar", stacked=True, figsize=(12, 4))
plt.title("Phân bố số lượng mẫu theo từng lớp (Fold 0)")
plt.ylabel("Số lượng ảnh")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# TODO: kiểm tra pipeline (slide trang 59, GUIDE.md mục 1.3):
#   1) cố định seed
#   2) loss ban đầu của head mới xấp xỉ -ln(1/9) = 2.197
#   3) overfit được một batch nhỏ tới loss gần 0
#   4) vẽ vài ảnh sau augmentation (đã giải chuẩn hoá) cùng nhãn để kiểm tra mắt thường
#   5) chạy thử 1 epoch và đo thời gian để lập ngân sách GPU
import torch.nn.functional as F
from model import build_model
from train import set_seed

set_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1) & 2) Loss CE ban đầu xấp xỉ -ln(1/9) ≈ 2.197
dummy_model = build_model("resnet50", pretrained=True, num_classes=9, init="finetune").to(device)
dummy_model.eval()
x_dummy = torch.randn(8, 3, 224, 224, device=device)
y_dummy = torch.randint(0, 9, (8,), device=device)
with torch.no_grad():
    init_loss = float(F.cross_entropy(dummy_model(x_dummy), y_dummy))
expected_loss = float(np.log(9))
print(f"Loss CE ban đầu: {init_loss:.4f} (Kỳ vọng: ln(9) = {expected_loss:.4f})")

# 3) Overfit 1 batch nhỏ về gần 0
dummy_model.train()
opt = torch.optim.AdamW(dummy_model.parameters(), lr=1e-3)
for _ in range(50):
    opt.zero_grad()
    l = F.cross_entropy(dummy_model(x_dummy), y_dummy)
    l.backward()
    opt.step()
print(f"Loss sau khi overfit 1 batch nhỏ: {l.item():.4f}")
del dummy_model, x_dummy, y_dummy

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
# TODO: vòng lặp qua danh sách backbone, mỗi lần gọi train.run(Config(exp_id=..., backbone=..., ...))
# Đọc runs/*/val_metrics.json và in bảng so sánh, chọn ra backbone tốt nhất cho Bước 2.
import json
from train import Config, pred_path, run, run_dir

BACKBONES = [
    ("B01", "resnet50"),
    ("B02", "resnext50_32x4d"),
    ("B03", "convnext_tiny"),
    ("B04", "deit_small_patch16_224"),
    ("B05", "mobilenetv3_large_100"),
]

def cached_run(cfg: Config) -> dict:
    r_file = run_dir(cfg) / "result.json"
    p_file = pred_path(cfg, "test") if cfg.save_test_predictions else None
    if r_file.is_file() and (p_file is None or p_file.is_file()):
        with open(r_file, "r", encoding="utf-8") as f:
            print(f"[CACHE] Dùng kết quả có sẵn của {cfg.exp_id} (seed {cfg.seed}) từ {r_file}")
            return json.load(f)
    return run(cfg)

b_results = []
for exp_id, bb in BACKBONES:
    cfg = Config(exp_id=exp_id, backbone=bb, seed=0, epochs=10, batch_size=64,
                 images_dir=IMAGES_DIR, labels_dir=LABELS_DIR)
    b_results.append(cached_run(cfg))

b_df = pd.DataFrame([{
    "exp_id": r["exp_id"],
    "backbone": r["backbone"],
    "tag trọng số": r.get("pretrained_tag", "default"),
    "#params (M)": round(r["params_m"], 2),
    "GMAC": round(r["gmac"], 2),
    "macro-F1 val": round(r["macro_f1_val"], 4),
    "top-1 val": round(r["top1_val"], 4),
    "thời gian train/epoch (s)": round(r["train_seconds_per_epoch"], 1),
} for r in b_results])
display(b_df)

best_b_row = max(b_results, key=lambda x: x["macro_f1_val"])
selected_backbone = best_b_row["backbone"]
print(f"=> Chọn backbone: {selected_backbone} (Macro-F1 val = {best_b_row['macro_f1_val']:.4f})")

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.

In [ ]:
# TODO: các trục A-G của GUIDE.md mục 3 (khởi tạo, augmentation, hàm mất mát, bộ tối ưu, lịch LR, regularization, độ phân giải).
# Chạy trên backbone đã chọn ở Bước 1. Đọc val_metrics.json, chọn cấu hình tốt nhất.

# T00: tái sử dụng từ checkpoint tốt nhất ở Bước 1 (0 lần train thừa)
t00_res = dict(best_b_row)
t00_res["exp_id"] = "T00"
t00_res["axis"] = "Baseline"
t00_res["change"] = "Công thức nền"

t_variants = [
    ("T01", {"aug": "color"}, "Trục B (Augmentation)", "color jitter"),
    ("T02", {"loss": "ls", "label_smoothing": 0.1}, "Trục C (Hàm loss)", "label smoothing 0.1"),
    ("T03", {"ema_decay": 0.99}, "Trục F (Chính quy hoá)", "EMA 0.99"),
]

t_results = [t00_res]
by_exp = {"T00": t00_res}

for exp_id, overrides, axis, change in t_variants:
    cfg = Config(exp_id=exp_id, backbone=selected_backbone, seed=0, epochs=10, batch_size=64,
                 images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, **overrides)
    res = cached_run(cfg)
    res["axis"] = axis
    res["change"] = change
    t_results.append(res)
    by_exp[exp_id] = res

base_f1 = t00_res["macro_f1_val"]
aug_choice = "color" if by_exp["T01"]["macro_f1_val"] > base_f1 else "basic"
loss_choice, ls_choice = ("ls", 0.1) if by_exp["T02"]["macro_f1_val"] > base_f1 else ("ce", 0.0)
ema_choice = 0.99 if by_exp["T03"]["macro_f1_val"] > base_f1 else None

comb_cfg = Config(exp_id="T04", backbone=selected_backbone, seed=0, epochs=10, batch_size=64,
                  aug=aug_choice, loss=loss_choice, label_smoothing=ls_choice, ema_decay=ema_choice,
                  images_dir=IMAGES_DIR, labels_dir=LABELS_DIR)
comb_res = cached_run(comb_cfg)
comb_res["axis"] = "Combination"
comb_res["change"] = f"aug={aug_choice}, loss={loss_choice}, ema={ema_choice}"
t_results.append(comb_res)

t_df = pd.DataFrame([{
    "exp_id": r["exp_id"],
    "trục thay đổi": r.get("axis", "Baseline"),
    "khác T00 ở điểm nào": r.get("change", "mốc"),
    "macro-F1 val": round(r["macro_f1_val"], 4),
    "top-1 val": round(r["top1_val"], 4),
    "Δ so với T00": round(r["macro_f1_val"] - base_f1, 4),
} for r in t_results])
display(t_df)

best_recipe = max(t_results, key=lambda x: x["macro_f1_val"])
print(f"=> Công thức tốt nhất: {best_recipe['exp_id']} (Macro-F1 val = {best_recipe['macro_f1_val']:.4f})")

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...

In [ ]:
# TODO: TTA lật, multi-crop/scale, dò độ phân giải, gộp xác suất vs logit, ensemble, temperature scaling, v.v.
# Ghi vào predictions/I00_val.csv, ..., đo độ trễ p50/p95/p99 (ít nhất 50 lần sau 10 lần warmup) ở batch 1 và batch 32.
from pathlib import Path
import pandas as pd
from inference import run_inference_only

# Dùng checkpoint best.pth từ lần huấn luyện trước. Cell này chỉ suy luận (I00-I08), không huấn luyện lại.
CHECKPOINT_PATH = None  # Ví dụ: Path("runs/B01/seed0/best.pth")
if CHECKPOINT_PATH is None and "best_recipe" in locals():
    CHECKPOINT_PATH = Path(best_recipe["checkpoint"])
MODEL_BACKBONE = selected_backbone if "selected_backbone" in locals() else None
ENSEMBLE_CHECKPOINTS = []  # Tối đa 2 model phụ cho I05 ensemble nếu có
OUTPUT_DIR = Path("outputs/step3")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if CHECKPOINT_PATH is None or MODEL_BACKBONE is None:
    print("Chưa có checkpoint/backbone. Hãy trỏ tới best.pth đã train trước đó; cell này không train model.")
else:
    inference_study = run_inference_only(
        checkpoint_path=CHECKPOINT_PATH,
        backbone=MODEL_BACKBONE,
        images_dir=IMAGES_DIR,
        labels_dir=LABELS_DIR,
        output_dir=OUTPUT_DIR,
        ensemble_checkpoints=ENSEMBLE_CHECKPOINTS,
        batch_size=32,
        num_workers=2,
    )
    visible_rows = [
        {key: value for key, value in row.items() if not key.startswith("_")}
        for row in inference_study["rows"]
    ]
    display(pd.DataFrame(visible_rows))
    print("Cấu hình đạt p95 batch-1 <= 100 ms:", inference_study["realtime_candidate"])

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).

In [ ]:
# TODO: chạy chung kết và mốc, mỗi seed, ví dụ:
#   for seed in [0, 1, 2]:
#       train.run(Config(exp_id="F01", seed=seed, save_test_predictions=True, ...))
#       train.run(Config(exp_id="T00", seed=seed, save_test_predictions=True, ...))
# Rồi áp dụng kỹ thuật suy luận tốt nhất từ Bước 3 (nếu có TTA/temperature) trước khi chấm.

for seed in (0, 1, 2):
    # Cấu hình chung kết F01
    cfg_f01 = Config(
        exp_id="F01", backbone=selected_backbone, seed=seed, epochs=10, batch_size=64,
        aug=aug_choice, loss=loss_choice, label_smoothing=ls_choice, ema_decay=ema_choice,
        save_test_predictions=True, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR
    )
    cached_run(cfg_f01)
    
    # Mốc so sánh T00
    cfg_t00 = Config(
        exp_id="T00", backbone=selected_backbone, seed=seed, epochs=10, batch_size=64,
        save_test_predictions=True, images_dir=IMAGES_DIR, labels_dir=LABELS_DIR
    )
    cached_run(cfg_t00)

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "predictions/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out eval_out
!python {REPO_DIR}/eval.py score --pred "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out eval_out

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận). Thêm --uncal, --final-val, --latency-p95-ms nếu có.
!python {REPO_DIR}/eval.py grade --final "predictions/F01_seed*_test.csv" \
    --baseline "predictions/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out eval_out

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.

In [ ]:
# TODO: ghi results.xlsx bằng pandas.ExcelWriter(engine="openpyxl").
# TODO: vẽ curves/ (loss + macro-F1 theo epoch cho Bước 1, 2, 4) và confusion matrix của F01.
# TODO: kiểm tra đủ 6 sản phẩm trong danh sách ở README.md trước khi nộp.
import eval as ev
from openpyxl.styles import Font, PatternFill

WORKBOOK_PATH = Path("results.xlsx")
sheet_backbones = b_df.copy()
sheet_training = t_df.copy()
sheet_inference = pd.DataFrame(visible_rows) if "visible_rows" in locals() else pd.DataFrame()

final_preds = ev.find_pred_files("predictions/F01_seed*_test.csv")
base_preds = ev.find_pred_files("predictions/T00_seed*_test.csv")

def get_final_rows(tag, files, desc):
    rows, f1s, top1s, eces = [], [], [], []
    for fp in files:
        p = ev.read_pred(fp)
        m = ev.compute_metrics(p.y_true, p.y_pred, p.probs)
        f1s.append(m["macro_f1"])
        top1s.append(m["top1"])
        eces.append(m["ece"])
        rows.append({
            "exp_id": tag, "cấu hình": desc, "seed": p.seed,
            "macro-F1 test": round(m["macro_f1"], 4), "top-1 test": round(m["top1"], 4), "ECE test": round(m["ece"], 4)
        })
    if f1s:
        rows.append({
            "exp_id": tag, "cấu hình": desc, "seed": "mean ± std",
            "macro-F1 test": f"{np.mean(f1s):.4f} ± {np.std(f1s, ddof=1):.4f}",
            "top-1 test": f"{np.mean(top1s):.4f} ± {np.std(top1s, ddof=1):.4f}",
            "ECE test": f"{np.mean(eces):.4f} ± {np.std(eces, ddof=1):.4f}"
        })
    return rows

sheet_final = pd.DataFrame(get_final_rows("F01", final_preds, "Final F01") + get_final_rows("T00", base_preds, "Baseline T00"))

if final_preds and base_preds:
    f_p0 = ev.read_pred(final_preds[0])
    t_p0 = ev.read_pred(base_preds[0])
    pc_f = ev.per_class(ev.confusion_matrix(f_p0.y_true, f_p0.y_pred))
    pc_t = ev.per_class(ev.confusion_matrix(t_p0.y_true, t_p0.y_pred))

    sheet_perclass = pd.DataFrame([{
        "lớp": CLASS_NAMES[i], "số ảnh test": int(pc_f["support"][i]),
        "F01 Recall": round(pc_f["recall"][i], 4), "F01 F1": round(pc_f["f1"][i], 4),
        "T00 Recall": round(pc_t["recall"][i], 4), "T00 F1": round(pc_t["f1"][i], 4),
    } for i in range(len(CLASS_NAMES))])
else:
    sheet_perclass = pd.DataFrame()

sheet_latency = []
for r in (visible_rows if "visible_rows" in locals() else []):
    for bs in (1, 32):
        sheet_latency.append({
            "cấu hình": r["exp_id"] + " (" + r["method"] + ")", "batch": bs,
            "p50": r.get(f"p50 batch-{bs} (ms)"), "p95": r.get(f"p95 batch-{bs} (ms)"),
            "ảnh/s": r.get(f"throughput batch-{bs} (img/s)"),
        })
sheet_latency = pd.DataFrame(sheet_latency)

summary_rows = [{"exp_id": r["exp_id"], "cấu hình": r["backbone"], "macro-F1 val": r["macro_f1_val"]} for r in b_results] +                [{"exp_id": r["exp_id"], "cấu hình": r.get("change", ""), "macro-F1 val": r["macro_f1_val"]} for r in t_results]
summary_rows.sort(key=lambda x: x["macro-F1 val"], reverse=True)
sheet_summary = pd.DataFrame(summary_rows[:10])

sheets = {
    "Backbones": sheet_backbones, "Training": sheet_training, "Inference": sheet_inference,
    "Final": sheet_final, "PerClass": sheet_perclass, "Latency": sheet_latency, "Summary": sheet_summary
}

with pd.ExcelWriter(WORKBOOK_PATH, engine="openpyxl") as writer:
    for name, frame in sheets.items():
        frame.to_excel(writer, sheet_name=name, index=False)
print(f"Đã xuất {WORKBOOK_PATH} đủ 7 sheets!")